# Занятие 26. Практика: ML-пайплайн на kNN (~90 мин)

Вы **пишете весь код сами**. Ячейку **«Дано»** не меняйте.

Главная модель занятия — **kNN** (п. 3 теории). Пройдём цикл из занятия 25: постановка → разведочный анализ train → разбиение → baseline → kNN → подбор `k` → анализ ошибок → test.

Термины и определения — в `workflow_theory.ipynb`. Здесь только практика.

Ориентир по времени указан у каждого задания. Если застряли — идите дальше и вернитесь позже.

---
## Дано: датасет

Встроенный в scikit-learn датасет диагностики опухоли: по числовым измерениям нужно определить, доброкачественная опухоль (класс 1) или злокачественная (класс 0). В `load_breast_cancer` класс 0 — `malignant`, класс 1 — `benign`.

Один объект — одна опухоль. Все признаки числовые и **в разных масштабах** — это важно для kNN.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
X_all = data.data
y_all = data.target

print('Объектов:', X_all.shape[0], '| признаков:', X_all.shape[1])
print('Доля класса 1:', round(y_all.mean(), 3))
X_all.iloc[:, :5].head()

---
## Задание 0. Постановка задачи (~7 мин)

По мотивам пп. 5 и 8 теории. Прежде чем обучать kNN, зафиксируйте постановку.

1. Создайте словарь `task_spec` с ключами: `объект`, `цель`, `тип_задачи`, `действие`.
2. Задайте `PRIMARY_METRIC` — основную метрику (`'recall'`, `'precision'`, `'f1'` или `'accuracy'`). Если выбираете recall — позже считайте долю найденных **злокачественных** с `pos_label=0`.
3. В markdown-ячейке ниже объясните: почему пропустить злокачественную (**FN**, п. 8) опаснее, чем лишний раз перепроверить доброкачественную (**FP**)?

**Критерий:** словарь заполнен; метрика выбрана до обучения модели.

*Почему для этой задачи важнее не пропустить злокачественную опухоль?*

(напишите ответ здесь)

---
## Задание 1. Импорты и константы (~5 мин)

Подключите всё необходимое:

- `train_test_split`;
- `StandardScaler`;
- `KNeighborsClassifier`;
- `DummyClassifier`;
- `accuracy_score`, `f1_score`, `recall_score`, `confusion_matrix`.

Константы: `RANDOM_STATE = 42`, `TEST_SIZE` и `VAL_SIZE` для разбиения **60% / 20% / 20%** (как в п. 4 теории).

**Критерий:** ячейка выполняется без ошибок.

---
## Задание 2. Разбиение train / validation / test (~8 мин)

1. Двумя вызовами `train_test_split` разделите `X_all`, `y_all` на train / validation / test.
2. Стратифицируйте по `y_all`.
3. Сохраните `X_train`, `X_val`, `X_test`, `y_train`, `y_val`, `y_test`.

**Критерий:** размеры `(341, 114, 114)`.

---
## Задание 3. Разведочный анализ train (~8 мин)

Исследуйте **только train** (не validation и не test):

1. Доля классов в `y_train`.
2. `describe()` для 3–4 признаков в `X_train`.
3. Сравните разброс (`std`) двух признаков с очень разными масштабами (например, `mean area` и `mean smoothness`).

В markdown ниже ответьте: почему разный масштаб признаков мешает kNN? (п. 3, пп. 13–14)

**Критерий:** все пункты выведены; ответ сформулирован.

*Почему разные масштабы признаков мешают kNN?*

(напишите ответ здесь)

---
## Задание 4. Baseline (~8 мин)

1. Обучите `DummyClassifier(strategy='most_frequent')` на train.
2. Посчитайте `accuracy` на validation → сохраните в `dummy_acc`.
3. Сравните с долей большего класса в train.

**Критерий:** `dummy_acc` посчитан; это планка, которую kNN должен превзойти.

---
## Задание 5. kNN без масштабирования (~8 мин)

1. Обучите `KNeighborsClassifier(n_neighbors=5)` на **исходных** (немасштабированных) `X_train`, `y_train`.
2. Посчитайте `accuracy` на validation → `acc_raw`.
3. Сравните `acc_raw` с baseline.

**Критерий:** `acc_raw` посчитан. Запомните это число для задания 6.

---
## Задание 6. kNN с масштабированием (~10 мин)

1. Обучите `StandardScaler` **только на train**, преобразуйте train / val / test → `X_train_s`, `X_val_s`, `X_test_s`.
2. Обучите `KNeighborsClassifier(n_neighbors=5)` на масштабированном train.
3. Посчитайте `accuracy` на validation → `acc_scaled`.
4. Сравните `acc_raw` (задание 5) и `acc_scaled`.

В markdown ниже: помогло ли масштабирование? На сколько?

**Критерий:** масштабированные матрицы получены; вывод сделан.

*Помогло ли масштабирование kNN?*

(напишите ответ здесь)

---
## Задание 7. Подбор числа соседей k (~12 мин)

`k` — гиперпараметр, его подбирают на validation.

1. Переберите `k` из `[1, 3, 5, 7, 11, 15, 21]`.
2. Для каждого обучите kNN на масштабированном train и посчитайте `accuracy` на validation.
3. Соберите результаты в `pd.DataFrame` (столбцы `k`, `val_acc`).
4. Найдите лучшее `k` → сохраните в `best_k`.

**Критерий:** таблица по всем `k`; `best_k` определён по validation (не по test).

---
## Задание 8. Анализ ошибок лучшего kNN (~12 мин)

1. Обучите kNN с `best_k` на масштабированном train.
2. Постройте `confusion_matrix` на validation (п. 17): строки — истинный класс, столбцы — прогноз.
3. Для положительного класса 0 (злокачественная) выпишите TP, FN, FP, TN.
4. Посчитайте `recall_score(..., pos_label=0)` — долю найденных злокачественных.
5. Разбейте объекты validation на две группы по медиане `mean radius` (крупные / мелкие) и сравните долю ошибок.

В markdown ниже: сколько злокачественных пропущено (FN), чему равен recall для класса 0 и в какой группе ошибок больше?

**Критерий:** матрица ошибок, recall класса 0 и таблица по группам выведены.

*Сколько FN для злокачественного класса 0, чему равен recall и где больше ошибок?*

(напишите ответ здесь)

---
## Задание 9. Финальный test и новые объекты (~10 мин)

**Test (один раз):** обучите kNN с `best_k` на масштабированном train, посчитайте `accuracy` на test. Test больше не трогайте.

**Новый объект:** возьмите первую строку `X_test`, преобразуйте тем же `scaler` и получите прогноз kNN. Сравните с настоящей меткой.

**Критерий:** test-accuracy посчитан один раз; для нового объекта получен прогноз.

---
## Итог

Вы прошли цикл из п. 2 теории на kNN. Подробная таблица этапов — в теоретическом ноутбуке.

kNN должен превосходить baseline (п. 11), а масштабирование и удачное `k` — улучшать validation. На следующих занятиях в этот каркас подставятся другие модели.
